# 01 — Загрузка чата в Qdrant с USER2-base

Индексирует только готовый `keep`-корпус из `messages_keep_with_attachment_text.csv`. Чанки — **350 токенов** токенизатора `deepvk/USER2-base` с перекрытием **50 токенов**. Для каждого чанка сохраняются чат, дата, автор, исходное сообщение и пути к файлам. PDF/JPEG-ссылки попадают в payload Qdrant. Повторный запуск обновляет точки по стабильным ID.

In [ ]:
# При необходимости в kernel этого ноутбука выполните:
# %pip install "sentence-transformers==5.2.2" "transformers==4.57.4" "qdrant-client>=1.19" pandas tqdm requests

In [ ]:
from pathlib import Path
import ast, json, uuid
import pandas as pd
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient, models

ROOT = next(
    parent for parent in (Path.cwd(), *Path.cwd().parents) if (parent / "data").is_dir()
)
DATA_CSV = ROOT / "data/processed/qwen_review/messages_keep_with_attachment_text.csv"
QDRANT_URL = "http://localhost:6333"
COLLECTION = "telegram_keep_user2_350tok_50overlap"
MODEL_NAME = "deepvk/USER2-base"
CHUNK_SIZE_TOKENS = 350
CHUNK_OVERLAP_TOKENS = 50
BATCH_SIZE = 64
assert DATA_CSV.exists(), f"Не найден файл: {DATA_CSV}"
print("Корпус:", DATA_CSV)

In [ ]:
client = QdrantClient(url=QDRANT_URL, timeout=30)
print("Qdrant:", client.get_collections())
model = SentenceTransformer(MODEL_NAME)
tokenizer = model.tokenizer
VECTOR_SIZE = model.get_sentence_embedding_dimension()
print("Модель загружена:", MODEL_NAME, "| размерность:", VECTOR_SIZE)


def parse_paths(value):
    if not isinstance(value, str) or not value.strip():
        return []
    try:
        result = json.loads(value)
    except Exception:
        try:
            result = ast.literal_eval(value)
        except Exception:
            return []
    return result if isinstance(result, list) else []


def chunk_text(text, size=CHUNK_SIZE_TOKENS, overlap=CHUNK_OVERLAP_TOKENS):
    text = str(text or "").strip()
    if not text:
        return []
    token_ids = tokenizer.encode(text, add_special_tokens=False)
    if len(token_ids) <= size:
        return [text]
    chunks = []
    start = 0
    while start < len(token_ids):
        end = min(start + size, len(token_ids))
        while end > start:
            part = tokenizer.decode(
                token_ids[start:end],
                skip_special_tokens=True,
                clean_up_tokenization_spaces=False,
            ).strip()
            # Decoding then re-encoding may change token boundaries; enforce the actual text limit.
            if len(tokenizer.encode(part, add_special_tokens=False)) <= size:
                break
            end -= 1
        if end <= start:
            raise ValueError("Токенизатор не смог собрать чанк в заданный лимит")
        if part:
            chunks.append(part)
        if end >= len(token_ids):
            break
        start = max(start + 1, end - overlap)
    return chunks


def absolute_paths(paths):
    return [str((ROOT / p).resolve()) for p in paths]


df = pd.read_csv(DATA_CSV).sort_values(
    ["date", "chat_name", "message_id"], kind="stable"
)
if "reviewer_decision" in df.columns:
    df = df[df.reviewer_decision.fillna("keep").eq("keep")]
else:
    df = df[df.qwen_label.eq("keep")]
df["flat_text"] = df.flat_text.fillna("").astype(str)
df = df[df.flat_text.str.strip().ne("")].copy()
print(f"Сообщений с текстом: {len(df):,}; даты: {df.date.min()} — {df.date.max()}")

In [ ]:
records = []
for row in df.to_dict("records"):
    paths = absolute_paths(parse_paths(row.get("file_paths")))
    pdf_paths = [p for p in paths if p.lower().endswith(".pdf")]
    image_paths = [p for p in paths if p.lower().endswith((".jpg", ".jpeg", ".png"))]
    parts = chunk_text(row["flat_text"])
    for i, part in enumerate(parts):
        records.append(
            {
                "record_id": str(row.get("record_id", "")),
                "chunk_index": i,
                "text": part,
                "chunk_token_count": len(
                    tokenizer.encode(part, add_special_tokens=False)
                ),
                "chat_name": str(row.get("chat_name", "")),
                "message_id": str(row.get("message_id", "")),
                "date": str(row.get("date", "")),
                "sender": str(row.get("sender", ""))
                if pd.notna(row.get("sender"))
                else "",
                "file_paths": paths,
                "pdf_paths": pdf_paths,
                "image_paths": image_paths,
                "source_input_file": str(row.get("input_file", "")),
            }
        )
assert all(r["chunk_token_count"] <= CHUNK_SIZE_TOKENS for r in records)
print(
    f"Подготовлено чанков: {len(records):,}; максимум {max(r['chunk_token_count'] for r in records)} токенов; параметры: {CHUNK_SIZE_TOKENS}/{CHUNK_OVERLAP_TOKENS}"
)
print("Фрагмент с PDF:", next((r for r in records if r["pdf_paths"]), "PDF-ссылок нет"))

## Создание коллекции и загрузка

Векторное расстояние — cosine. Для чанков используется токенизатор той же модели, что и для эмбеддингов. Коллекция `telegram_keep_user2_base_v1`, ранее созданная с размером чанка в символах, удаляется, чтобы в Qdrant не осталось ошибочной индексации.

In [ ]:
existing = {c.name for c in client.get_collections().collections}
if COLLECTION not in existing:
    client.create_collection(
        collection_name=COLLECTION,
        vectors_config=models.VectorParams(
            size=VECTOR_SIZE, distance=models.Distance.COSINE
        ),
    )
else:
    info = client.get_collection(COLLECTION)
    vec = info.config.params.vectors
    if isinstance(vec, dict):
        dims = {name: cfg.size for name, cfg in vec.items()}
        assert any(size == VECTOR_SIZE for size in dims.values()), (
            f"Неверная размерность коллекции: {dims}"
        )
    else:
        assert vec.size == VECTOR_SIZE, f"Неверная размерность: {vec.size}"

for start in tqdm(range(0, len(records), BATCH_SIZE), desc="Векторизация и upsert"):
    batch = records[start : start + BATCH_SIZE]
    texts = [r["text"] for r in batch]
    vectors = model.encode(
        texts,
        prompt_name="search_document",
        batch_size=BATCH_SIZE,
        show_progress_bar=False,
        convert_to_numpy=True,
    )
    points = []
    for rec, vector in zip(batch, vectors):
        point_id = str(
            uuid.uuid5(
                uuid.NAMESPACE_URL,
                f"{rec['record_id']}::{rec['chunk_index']}::{CHUNK_SIZE_TOKENS}tokens::{CHUNK_OVERLAP_TOKENS}overlap",
            )
        )
        payload = {
            **rec,
            "model": MODEL_NAME,
            "chunk_size_tokens": CHUNK_SIZE_TOKENS,
            "chunk_overlap_tokens": CHUNK_OVERLAP_TOKENS,
        }
        points.append(
            models.PointStruct(id=point_id, vector=vector.tolist(), payload=payload)
        )
    client.upsert(collection_name=COLLECTION, points=points, wait=True)
print(client.get_collection(COLLECTION))

## Проверка нескольких точек

In [ ]:
sample = client.scroll(
    collection_name=COLLECTION, limit=5, with_payload=True, with_vectors=False
)[0]
for p in sample:
    print(
        p.payload.get("date"),
        p.payload.get("chat_name"),
        p.payload.get("text", "")[:140].replace("\n", " "),
    )
    if p.payload.get("pdf_paths"):
        print("PDF:", p.payload["pdf_paths"])